# Experiment

## Import libraries

In [24]:
import pandas as pd
import glob
import os

# Folder where your CSV files are located
folder_path = ""  # Add the path to your folder here

if os.path.isfile("Gold_Futures_Combined.csv"):
    os.remove("Gold_Futures_Combined.csv")

file_paths = glob.glob(os.path.join(folder_path, "*.csv"))

dfs = []
for file in file_paths:
    df = pd.read_csv(file)
    
    # Convert 'Date' to datetime
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y", errors='coerce')
    
    # Clean numeric columns: remove commas and symbols, then convert to float
    for col in ["Price", "Open", "High", "Low"]:
        df[col] = df[col].astype(str).str.replace(',', '').astype(float)
    
    # Volume: remove 'K', 'M' and convert to float with multiplier
    def parse_volume(val):
        if pd.isna(val):
            return None
        val = str(val).strip().replace(',', '')
        if val.endswith('K'):
            return float(val[:-1]) * 1_000
        elif val.endswith('M'):
            return float(val[:-1]) * 1_000_000
        else:
            return float(val)


    df["Vol."] = df["Vol."].apply(parse_volume)

    # Change %: remove '%' and convert to float
    df["Change %"] = df["Change %"].astype(str).str.replace('%', '').astype(float)

    dfs.append(df)

# Combine and sort
full_df = pd.concat(dfs, ignore_index=True)
full_df = full_df.sort_values('Date').reset_index(drop=True)

# Save to CSV
full_df.to_csv('Gold_Futures_Combined.csv', index=False)

# Optional: Display the final DataFrame
full_df.columns

Index(['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %'], dtype='object')